<a href="https://colab.research.google.com/github/weagan/Energy-Based-Models-Predictive-Coding/blob/main/Iterative_predictive_coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predictive Coding Network: Mathematical Formulation

The neural network implemented in your code is a **Predictive Coding** model that minimizes a local "Energy" (or "Surprise") function. Unlike standard networks, it avoids global backpropagation by using local error signals and iterative inference.

---

### 1. The Energy Function ($E$)
The core objective is to minimize the total energy $E$ for a given input $v$, defined as the sum of squared prediction errors across layers:

$$E = \|v - \sigma(W_{td}h)\|^2 + \|h - W_{bu}v\|^2$$

**Variables:**
* $v$: The **visible** layer (input vector).
* $h$: The **hidden** layer (latent representation).
* $W_{td}$: **Top-down** (generative) weights.
* $W_{bu}$: **Bottom-up** (recognition) weights.
* $\sigma$: The **sigmoid** activation function.

---

### 2. Prediction Errors
The model explicitly tracks the discrepancy between layers using two error vectors:

* **Visible Prediction Error ($e_v$):** The difference between the actual input and the top-down prediction.
    $$e_v = v - \sigma(W_{td}h)$$
* **Hidden Prediction Error ($e_h$):** The difference between the current hidden state and the state suggested by the bottom-up weights.
    $$e_h = h - (W_{bu}v)$$

---

### 3. Iterative Inference
In this architecture, the hidden state $h$ is not calculated in a single pass. Instead, it is found by performing gradient descent on the Energy function $E$:

$$\Delta h \propto -\frac{\partial E}{\partial h}$$

Using the chain rule as implemented in the `inference` function, the gradient is:
$$\frac{\partial E}{\partial h} = e_h - W_{td}^T (e_v \odot \sigma'(W_{td}h))$$

> **Note:** $\odot$ denotes the element-wise Hadamard product, and $\sigma'$ is the derivative of the sigmoid function.

---

### 4. Local Learning Rules
Weights are updated using **local** error signals rather than backpropagating through the entire graph. The update is the outer product ($\otimes$) of the error and the adjacent layer's activation:

**Top-Down Update (Generative):**
$$W_{td} \leftarrow W_{td} + \eta ((e_v \odot \sigma') \otimes h)$$

**Bottom-Up Update (Recognition):**
$$W_{bu} \leftarrow W_{bu} + \eta (e_h \otimes v)$$

* $\eta$ represents the learning rate (`LR_W`).

1. The Hadamard Product ($\odot$)
If $a = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$ and $b = \begin{bmatrix} 3 \\ 4 \end{bmatrix}$, then $a \odot b = \begin{bmatrix} 1 \times 3 \\ 2 \times 4 \end{bmatrix} = \begin{bmatrix} 3 \\ 8 \end{bmatrix}$.
2. The Outer Product ($\otimes$)
If $e_h = \begin{bmatrix} A \\ B \end{bmatrix}$ and $v = \begin{bmatrix} X & Y & Z \end{bmatrix}$, then:$$e_h \otimes v = \begin{bmatrix} AX & AY & AZ \\ BX & BY & BZ \end{bmatrix}$$

In [1]:
"""
Predictive Coding Demo  —  No Backprop
=======================================
Energy per sample:
    E = ‖v - sigmoid(W_td @ h)‖²  +  ‖h - W_bu @ v‖²

    e_v = v - sigmoid(W_td @ h)   visible prediction error
    e_h = h - W_bu @ v            hidden  prediction error

Inference  : gradient descent on E w.r.t. h  (iterative, no autograd)
Learning   : gradient descent on E w.r.t. W_td, W_bu  (local, no backprop)

Dataset    : 4 structured binary prototypes + noise → real structure to learn
"""

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# ── activations ───────────────────────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def tanh_prime(x):
    t = np.tanh(x)
    return 1.0 - t * t

# ── dataset ───────────────────────────────────────────────────────────────────
# 4 linearly-independent binary prototypes in 8 bits.
# Noisy copies give the model real structure to compress.
PROTOTYPES = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 1],
], dtype=float)

def make_dataset(n=400, p_flip=0.10):
    X = []
    for _ in range(n):
        proto = PROTOTYPES[np.random.randint(len(PROTOTYPES))]
        noise = np.random.rand(len(proto)) < p_flip
        X.append(np.abs(proto - noise.astype(float)))
    return np.array(X)

data = make_dataset()

def get_batch(batch_size=32):
    idx = np.random.choice(len(data), batch_size, replace=False)
    return data[idx]

# ── network ───────────────────────────────────────────────────────────────────
INPUT_DIM  = 8
HIDDEN_DIM = 4          # matches number of prototypes → compact representation

W_td = np.random.randn(INPUT_DIM,  HIDDEN_DIM) * 0.1  # top-down  (generative)
W_bu = np.random.randn(HIDDEN_DIM, INPUT_DIM)  * 0.1  # bottom-up (recognition)

LR_H = 0.2     # inference (h) step size
LR_W = 0.001   # weight learning rate

# ── inference  ────────────────────────────────────────────────────────────────
def inference(v, steps=50):
    """
    Iteratively minimise E w.r.t. h_pre by gradient descent.

    Full chain rule:
        dE/dh_pre = dE/dh · tanh'(h_pre)
        dE/dh     = -W_td.T @ (e_v · sigmoid'(W_td @ h))  +  e_h
    """
    h_pre = np.zeros(HIDDEN_DIM)

    for _ in range(steps):
        h      = np.tanh(h_pre)
        v_pred = sigmoid(W_td @ h)
        sig_p  = v_pred * (1.0 - v_pred)   # sigmoid derivative

        e_v = v - v_pred
        e_h = h - (W_bu @ v) #  (4x8) * (8x1) = (4x1).

        # gradient of E w.r.t. h, then chain through tanh
        dEdh_pre = (-W_td.T @ (e_v * sig_p) + e_h) * tanh_prime(h_pre)
        h_pre   -= LR_H * dEdh_pre

    # final state
    h      = np.tanh(h_pre)
    v_pred = sigmoid(W_td @ h) #  (8x4) * (4x1) = (8x1)
    e_v    = v - v_pred
    e_h    = h - (W_bu @ v)
    return h, e_v, e_h

# ── weight update  ────────────────────────────────────────────────────────────
def update_weights(v, h, e_v, e_h):
    """
    Gradient descent on E w.r.t. each weight matrix.

    dE/dW_td = -outer(e_v · sigmoid'(W_td @ h),  h)
    dE/dW_bu = -outer(e_h, v)

    → subtract gradient = add correction term.
    Both updates are *local*: each uses only adjacent-layer signals.
    No backprop through layers.
    """
    global W_td, W_bu
    v_pred = sigmoid(W_td @ h)
    sig_p  = v_pred * (1.0 - v_pred)

    W_td += LR_W * np.outer(e_v * sig_p, h)   # local: visible error × hidden act
    W_bu += LR_W * np.outer(e_h, v)           # local: hidden  error × visible act

    # mild clipping for numerical safety
    W_td = np.clip(W_td, -3.0, 3.0)
    W_bu = np.clip(W_bu, -3.0, 3.0)

# ── training loop  ────────────────────────────────────────────────────────────
N_EPOCHS = 300
losses   = []

for epoch in range(N_EPOCHS):
    batch      = get_batch(32)
    epoch_loss = 0.0

    for v in batch:
        h, e_v, e_h = inference(v)
        epoch_loss  += float(np.sum(e_v**2) + np.sum(e_h**2))
        update_weights(v, h, e_v, e_h)

    losses.append(epoch_loss / len(batch))

    if epoch % 50 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:3d}  |  Loss: {losses[-1]:.4f}")

# ── loss plot  ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color="steelblue", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Energy (per sample)")
ax.set_title("Predictive Coding — Training Loss (No Backprop)")
fig.tight_layout()
fig.savefig("loss_curve.png", dpi=130)
plt.close()
print("\nLoss curve saved → loss_curve.png")

# ── reconstruction test  ──────────────────────────────────────────────────────
print("\n── Reconstruction test (clean prototypes) ──")
test_samples = make_dataset(n=8, p_flip=0.0)   # clean, no noise

correct_bits = 0
total_bits   = 0

for v in test_samples:
    h, _, _  = inference(v, steps=80)
    v_pred   = sigmoid(W_td @ h)
    v_binary = (v_pred > 0.5).astype(float)

    match = (v_binary == v)
    correct_bits += int(match.sum())
    total_bits   += len(v)

    raw_str = "[" + ", ".join(f"{x:.2f}" for x in v_pred) + "]"
    result  = "PERFECT" if match.all() else f"{int(match.sum())}/8 bits"
    print(f"  Input:   {v.astype(int).tolist()}")
    print(f"  Output:  {v_binary.astype(int).tolist()}   raw={raw_str}")
    print(f"  → {result}\n")

print(f"Bit accuracy: {100 * correct_bits / total_bits:.1f}%")

Epoch   0  |  Loss: 1.9785
Epoch  50  |  Loss: 1.9688
Epoch 100  |  Loss: 1.9289
Epoch 150  |  Loss: 1.9064
Epoch 200  |  Loss: 1.7887
Epoch 250  |  Loss: 1.6951
Epoch 299  |  Loss: 1.5024

Loss curve saved → loss_curve.png

── Reconstruction test (clean prototypes) ──
  Input:   [0, 1, 0, 1, 0, 1, 0, 1]
  Output:  [0, 1, 0, 1, 0, 1, 0, 1]   raw=[0.40, 0.54, 0.41, 0.60, 0.42, 0.55, 0.37, 0.63]
  → PERFECT

  Input:   [0, 0, 0, 0, 1, 1, 1, 1]
  Output:  [0, 0, 0, 0, 1, 1, 1, 1]   raw=[0.36, 0.36, 0.40, 0.43, 0.56, 0.62, 0.60, 0.62]
  → PERFECT

  Input:   [0, 1, 0, 1, 0, 1, 0, 1]
  Output:  [0, 1, 0, 1, 0, 1, 0, 1]   raw=[0.40, 0.54, 0.41, 0.60, 0.42, 0.55, 0.37, 0.63]
  → PERFECT

  Input:   [0, 1, 0, 1, 0, 1, 0, 1]
  Output:  [0, 1, 0, 1, 0, 1, 0, 1]   raw=[0.40, 0.54, 0.41, 0.60, 0.42, 0.55, 0.37, 0.63]
  → PERFECT

  Input:   [0, 0, 0, 0, 1, 1, 1, 1]
  Output:  [0, 0, 0, 0, 1, 1, 1, 1]   raw=[0.36, 0.36, 0.40, 0.43, 0.56, 0.62, 0.60, 0.62]
  → PERFECT

  Input:   [1, 0, 1, 0, 1, 0, 

In [2]:
import numpy as np

rank_W_td = np.linalg.matrix_rank(W_td)
rank_W_bu = np.linalg.matrix_rank(W_bu)

print(f"Rank of W_td: {rank_W_td}")
print(f"Rank of W_bu: {rank_W_bu}")

Rank of W_td: 4
Rank of W_bu: 4
